# Mar 10/11 · 17:30 — TP v0.10 : requests robuste, API GitLab, webhook

Mar 10/11, TP de 17:55 à 18:35. Correction à 18:35, TP fini ou non ; à 18:45, lancement du projet de groupe. La référence v0.10 est publiée à 19:00.

## Objectif

`fleetcheck` devient l'outil de la démo du 12/10 :

- une `Session` pour tout le run, et de nouveaux essais sur les erreurs passagères, avec une pause qui double ;
- la version de chaque service de prod est comparée à la dernière release publiée sur l'API GitLab ;
- quand le parc est `down`, une alerte part vers le webhook dont l'URL est dans l'environnement.

## Mode d'emploi du notebook

- Copiez ce notebook à la racine de votre dépôt, à côté de `fleetcheck.py`. Le code s'écrit dans `fleetcheck.py`, ouvert dans un onglet de JupyterLab ; le notebook sert à essayer et à vérifier chaque étape.
- **Le serveur simulé doit tourner** : `python simserver.py` dans un terminal de JupyterLab, comme en v0.9. Il reçoit aussi l'alerte et l'affiche dans ce terminal.
- Exécutez d'abord la cellule d'outils ci-dessous. Chaque vérification affiche `OK`, ou `KO` avec ce qui était attendu.
- `%run -n fleetcheck.py` charge les fonctions du fichier enregistré sans exécuter le bloc `if __name__ == "__main__":`. N'appelez pas `main()` dans le notebook.
- `run(...)` lance le script comme dans un terminal et renvoie `r.stdout`, `r.stderr` et `r.returncode`. Avec les nouveaux essais, un run complet dure environ 16 s : les vérifications de l'étape 3 utilisent un inventaire de deux lignes, plus rapide.
- La cellule qui pose `FLEETCHECK_WEBHOOK_URL` la retire à la fin : aucune URL ne reste dans le notebook rendu.
- **Prédire** et **Justifier** marquent les moments clés du TP. Répondez dans la cellule `# Ma réponse :` qui suit, **avant** d'exécuter la suite.
- Rendu en fin de TP : menu *Kernel > Restart Kernel and Run All Cells*, serveur simulé lancé, puis commit et push des mêmes fichiers qu'en v0.9 et de ce notebook, sans `reports/`.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args):
    """Lance fleetcheck.py comme dans un terminal (vu en v0.8 : subprocess)."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r

Port du serveur simulé : `8000` par défaut. Si vous l'avez lancé avec `--base-port 9000`, mettez `BASE = 9000` ici avant d'exécuter la suite : toutes les URL des cellules en dépendent. Dans ce cas, changez aussi le port de `releases_url` dans `config.json` (`9000`) et ajoutez `"--inventory", "servers-sim.csv"` aux `run()` sur l'inventaire complet (voir le TP v0.9 pour écrire `servers-sim.csv`).

In [ ]:
BASE = 8000

## Point de départ

Votre `fleetcheck.py` de v0.9, avec `config.json`, `requirements.txt`, `.gitignore` et les deux inventaires.

En retard ou en panne : copiez dans votre dépôt les fichiers de `reference/` au tag `v0.9` du dépôt public.

`config.json` contient depuis v0.3 les réglages de cette séance : `attempts` (3), `backoff_s` (0,5), `drift_envs` (`["prod"]`) et `releases_url`, qui pointe vers la fausse API GitLab du serveur simulé.

Vérifiez avant de commencer :

In [ ]:
import json
import os
import time
from pathlib import Path

import requests

%run -n fleetcheck.py
check((callable(check_health), callable(check_service)), (True, True))
config = json.loads(Path("config.json").read_text(encoding="utf-8"))
check((config["attempts"], config["backoff_s"], config["drift_envs"]), (3, 0.5, ["prod"]))
check(config["releases_url"], f"http://127.0.0.1:{BASE}/api/v4/projects/fleetcheck/releases")

## Étapes

| Fin | Étape |
|---|---|
| 18:10 | 1. `Session` et nouveaux essais dans `check_health` |
| 18:25 | 2. Dernière release et écart de version |
| 18:35 | 3. Alerte par webhook |

À 18:15, si plus d'un tiers de la salle n'a pas de retry qui passe `8005` à `200`, l'enseignant écrit la boucle au tableau ; l'étape 2 se limite à `latest_release`.

### 1. Session et nouveaux essais — fin 18:10

- `main()` crée `session = requests.Session()` et la passe à `check_service`, qui la passe à `check_health`. Nouvelles signatures : `check_health(session, url, config)` et `check_service(server, session, config, latest)` (`latest` sert à l'étape 2 : passez `None` en attendant).
- `check_health` fait jusqu'à `config["attempts"]` essais. Un essai est en échec sur `requests.Timeout`, `requests.ConnectionError` ou un code `>= 500` ; tout autre code (`2xx`, `4xx`) renvoie le résultat aussitôt.
- Entre deux essais, pause de `config["backoff_s"]`, doublée à chaque fois (`time.sleep`) : 0,5 s, puis 1 s. Pas de pause après le dernier essai.
- `attempts` du résultat = nombre d'essais faits. Après le dernier échec, on renvoie le résultat du dernier essai. Délai de chaque essai : `config["timeout_s"]`.
- Dans `check_service`, la raison d'un `down` HTTP prend le nombre d'essais : `HTTP 503 après 3 essai(s)`, `timeout après 3 essai(s)`, `connexion impossible après 3 essai(s)`.
- Facultatif : un `log.debug` par essai en échec.

**Justifier** — Pourquoi pas de nouvel essai sur un `404` ?

In [ ]:
# Ma réponse :

Vérification : `worker-1` finit par répondre, `api-2` échoue 3 fois avec les pauses, un `404` ne se réessaie pas, un port fermé si.

In [ ]:
%run -n fleetcheck.py
session = requests.Session()
r = check_health(session, f"http://127.0.0.1:{BASE + 5}/health", config)
check((r["http_status"], 1 <= r["attempts"] <= 3), (200, True))
debut = time.monotonic()
r = check_health(session, f"http://127.0.0.1:{BASE + 4}/health", config)
check((r["http_status"], r["attempts"], time.monotonic() - debut >= 1.5), (503, 3, True))
r = check_health(session, f"http://127.0.0.1:{BASE + 1}/inconnu", config)
check((r["http_status"], r["attempts"]), (404, 1))
r = check_health(session, f"http://127.0.0.1:{BASE + 9}/health", config)
check((r["error"], r["attempts"]), ("connexion impossible", 3))

### 2. Dernière release et écart de version — fin 18:25

`latest_release(session, url, token, timeout_s)` :

- `GET` sur `url` avec l'en-tête `PRIVATE-TOKEN: <token>` si `token` n'est pas `None` (`headers={"PRIVATE-TOKEN": token} if token else {}`) ; `main()` lui passe `config["releases_url"]` et `os.environ.get("FLEETCHECK_GITLAB_TOKEN")` ;
- la réponse est une liste de releases, la plus récente en premier : renvoyez son `tag_name` sans le `v` de tête (`removeprefix("v")`) ;
- en cas d'échec (`requests.RequestException`, `raise_for_status()`, JSON inattendu, liste vide) : `WARNING` dans le log, et `None`.

`main()` l'appelle une fois, avant la boucle, et passe le résultat à `check_service`. Dans `check_service`, après `/health` et seulement si le service n'est pas `down` :

- `GET <ip>:<port>/version`, dont le corps vaut `{"service": "web-1", "version": "2.3.1"}` ; échec = `WARNING` et pas d'écart ;
- si l'env du serveur est dans `config["drift_envs"]` et que la version et `latest` sont connues et différentes : `degraded`, raison `version 2.3.0 ≠ 2.3.1` ;
- rangez `{"value": <version ou None>, "drift": <True, False ou None>}` sous la clé `version`.

**Prédire** — `worker-1` : version `2.3.0`, dernière release `2.3.1`, `/health` en `200` au troisième essai. Si c'était le seul service de l'inventaire, quels statut et code de sortie ?

In [ ]:
# Ma réponse :

Vérification :

In [ ]:
%run -n fleetcheck.py
session = requests.Session()
latest = latest_release(session, config["releases_url"], None, 2)
check(latest, "2.3.1")
worker = {"name": "worker-1", "env": "prod", "ip": "127.0.0.1", "port": BASE + 5, "role": "worker"}
stg = {"name": "web-stg-1", "env": "staging", "ip": "127.0.0.1", "port": BASE + 6, "role": "web"}
r = check_service(worker, session, config, latest)
check((r["status"], r["reasons"]), ("degraded", ["version 2.3.0 ≠ 2.3.1"]))
r = check_service(stg, session, config, latest)
check((r["status"], r["reasons"]), ("ok", []))

### 3. Alerte par webhook — fin 18:35

`send_alert(session, webhook_url, report)` :

- `session.post(webhook_url, json={"content": message}, timeout=5)`, puis `raise_for_status()` ;
- `message` : `fleetcheck : N service(s) down`, puis une ligne `- <nom> (<env>) : <raisons>` par service `down` ;
- succès : log `INFO alerte envoyée` ; `requests.RequestException` : log `WARNING échec de l'envoi de l'alerte : <erreur>`. Le code de sortie ne change pas.

Dans `main()`, après l'écriture du rapport : si le statut global vaut `down`, lire `FLEETCHECK_WEBHOOK_URL` ; définie, `send_alert` ; absente, `INFO FLEETCHECK_WEBHOOK_URL absent : pas d'alerte`. Le message de v0.6 en début de `main()` disparaît.

**Justifier** — Pourquoi l'échec de l'alerte ne change-t-il pas le code de sortie ?

In [ ]:
# Ma réponse :

Vérification, cellule fournie : elle écrit `servers-alerte.csv` (`web-1` sain, `cache-1` sur un port fermé, donc `down`), pose la variable, lance trois runs, puis retire la variable. Au premier run, le message apparaît dans le terminal du serveur simulé.

In [ ]:
Path("servers-alerte.csv").write_text(
    "name,env,ip,port,role\n"
    f"web-1,prod,127.0.0.1,{BASE + 1},web\n"
    f"cache-1,prod,127.0.0.1,{BASE + 9},cache\n", encoding="utf-8")
try:
    os.environ["FLEETCHECK_WEBHOOK_URL"] = f"http://127.0.0.1:{BASE}/webhook"
    r = run("--inventory", "servers-alerte.csv", "-v")
    check(("INFO alerte envoyée" in r.stderr.splitlines(), r.returncode), (True, 1))

    os.environ["FLEETCHECK_WEBHOOK_URL"] = f"http://127.0.0.1:{BASE}/inconnu"
    r = run("--inventory", "servers-alerte.csv", "-v")
    echec = [l for l in r.stderr.splitlines() if l.startswith("WARNING échec de l'envoi de l'alerte")]
    check((len(echec), r.returncode), (1, 1))

    del os.environ["FLEETCHECK_WEBHOOK_URL"]
    r = run("--inventory", "servers-alerte.csv", "-v")
    check(("alerte envoyée" in r.stderr, "échec de l'envoi" in r.stderr, r.returncode), (False, False, 1))
finally:
    os.environ.pop("FLEETCHECK_WEBHOOK_URL", None)

## Critère final

Inventaire complet, environ 16 s : c'est le résultat de la démo de l'ouverture.

```text
ok        web-1        prod
degraded  web-2        prod     latence 1.51 s > 1.0 s
ok        api-1        prod
down      api-2        prod     HTTP 503 après 3 essai(s)
degraded  worker-1     prod     version 2.3.0 ≠ 2.3.1
ok        web-stg-1    staging
down      api-stg-1    staging  timeout après 3 essai(s)
down      cache-1      prod     connexion impossible après 3 essai(s)
down      db-1         prod     ping KO

DOWN : 3 ok, 2 degraded, 4 down, 0 invalide(s)
```

Les centièmes de `web-2` varient d'un run à l'autre : la vérification compare les statuts et les raisons, pas la sortie texte.

In [ ]:
r = run("--format", "json")
rapport = json.loads(r.stdout)
services = {s["name"]: s for s in rapport["services"]}
check({nom: s["status"] for nom, s in services.items()},
      {"web-1": "ok", "web-2": "degraded", "api-1": "ok", "api-2": "down", "worker-1": "degraded",
       "web-stg-1": "ok", "api-stg-1": "down", "cache-1": "down", "db-1": "down"})
check({nom: s["reasons"] for nom, s in services.items() if nom != "web-2"},
      {"web-1": [], "api-1": [], "api-2": ["HTTP 503 après 3 essai(s)"], "worker-1": ["version 2.3.0 ≠ 2.3.1"],
       "web-stg-1": [], "api-stg-1": ["timeout après 3 essai(s)"],
       "cache-1": ["connexion impossible après 3 essai(s)"], "db-1": ["ping KO"]})
check((rapport["latest_release"], rapport["summary"]["status"], r.returncode), ("2.3.1", "down", 1))

## Pour aller plus loin

- Une vraie API GitLab, sans token, sur un projet public : remplacez le temps d'un run `releases_url` par `https://gitlab.com/api/v4/projects/gitlab-org%2Fgitlab-runner/releases?per_page=1` et regardez le `WARNING` d'écart de version (le réseau de l'école doit laisser passer `gitlab.com`).
- Un webhook Discord réel : créez-en un dans un salon de test (*Paramètres du salon > Intégrations > Webhooks*), posez son URL dans `FLEETCHECK_WEBHOOK_URL` dans votre terminal, jamais dans un fichier du dépôt, et lancez `fleetcheck`.